# 📊 Notebook 5e: Master Ablation Study & Results Summary Aggregation
This notebook evaluates all trained model checkpoints from `nb2`, `nb3`, and `nb5a-d` to generate the master academic comparison table.
* **Input Checkpoints**: `best.pt` files from all ablation and production runs.
* **Outputs**: Master Markdown table with Mask mAP50, Mask mAP50-95, Box mAP50, and Precision/Recall.


In [ ]:
!mkdir -p configs utils distillation scripts checkpoints data/datasets


In [ ]:
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas


In [ ]:
# Auto-fetch project modules to ensure 100% self-contained execution on Kaggle
!git clone https://github.com/shahin1717/crackdistill.git repo_code || true
!cp -r repo_code/distillation repo_code/utils repo_code/configs repo_code/scripts .


In [ ]:
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

ablation_runs = [
    ("Baseline (No KD Fine-tune)", "runs/crack_distill_baseline_finetune_instance_seg_yolo11n-seg/weights/best.pt"),
    ("Full KD (Box Prompts)", "runs/crack_distill_full_kd_box_instance_seg_yolo11n-seg/weights/best.pt"),
    ("Full KD (Box + Centroid)", "runs/crack_distill_full_kd_centroid_instance_seg_yolo11n-seg/weights/best.pt"),
    ("Ablation 1: w/o Mask KL (nb5a)", "runs/crack_distill_ablation_no_mask_kd_instance_seg_yolo11n-seg/weights/best.pt"),
    ("Ablation 2: w/o Feature MSE (nb5b)", "runs/crack_distill_ablation_no_feature_instance_seg_yolo11n-seg/weights/best.pt"),
    ("Ablation 3: w/o Boundary BCE (nb5c)", "runs/crack_distill_ablation_no_boundary_instance_seg_yolo11n-seg/weights/best.pt"),
    ("Ablation 4: Full SegHead Freeze (nb5d)", "runs/crack_distill_ablation_seghead_frozen_instance_seg_yolo11n-seg/weights/best.pt"),
]

data_yaml = "data/datasets/crack500_yolo/dataset.yaml"
results = []
for name, ckpt in ablation_runs:
    ckpt_path = Path(ckpt)
    if ckpt_path.exists():
        m = YOLO(str(ckpt_path))
        res = m.val(data=data_yaml, split="val")
        results.append({
            "Ablation / Model Variant": name,
            "Mask mAP50": res.seg.map50,
            "Mask mAP50-95": res.seg.map,
            "Box mAP50": res.box.map50,
            "Box mAP50-95": res.box.map,
        })
    else:
        print(f"Skipping {name}: checkpoint {ckpt} not found.")

df = pd.DataFrame(results)
print("\n" + "="*70)
print("🔬 MASTER ABLATION STUDY RESULTS SUMMARY")
print("="*70)
print(df.to_string(index=False))
